# Anonymous review adaptation

This notebook is adapted from the research implementation. Model labels and output names use **Adjustment**. Personal paths are replaced by a local configuration. Hyperparameters, data screens, and estimation logic are otherwise retained. The MiniLM benchmark uses the same external-snapshot alignment as the other encoders.

**Required:** a locally authorized analysis sample and completed embedding export. These inputs are not bundled. This adaptation has passed static checks and CPU component tests; full neural training has not been rerun in the packaging environment. New code identities create separate checkpoints; original results remain unchanged.


In [ ]:
from pathlib import Path
import runpy
_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in _candidates if (p/'src'/'project_config.py').is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter from the repository root or its notebooks folder.")
_config_api = runpy.run_path(str(REPO_ROOT/'src'/'project_config.py'))
REPO_CONFIG = _config_api['load_project_config'](REPO_ROOT)
print('Repository configuration loaded. Input files are read locally; nothing is uploaded.')


# Industry heterogeneity — Llama Adjustment-RF + RF R-learner (Windows)

Standalone notebook. Reads the existing matched financial analysis table and completed Llama report vectors; trains new Adjustment, RF nuisance and heterogeneous-effect models. Does not require any earlier benchmark checkpoint. Does not extract text, embed text, download SEC data or rebuild Compustat.

**What you receive:** a held-out estimated effect for each firm/report period; a SIC2 industry mean-effect table; a within-industry distribution for each sufficiently supported industry; and annual descriptive patterns. Positive effects mean higher predicted next-fiscal-year ROA as competition language increases. One effect unit is **one ROA percentage point per +1 percentage point of competition sentence share**. These are conditional slope estimates, not observed individual causal effects or automatically identified binary ATEs.

This is exploratory. The earlier simulation showed remaining bias and poor coverage under strong confounding. This extension does not repair identification, filing timing, company identity or extraction issues. Do not interpret estimated distribution spread as proven causal heterogeneity or as an uncertainty interval.


## Step 1 — Configuration
Use your existing CUDA/PyTorch environment; the additional HTE layer uses scikit-learn's RF and needs **no EconML installation**. If an import is missing, install only the missing package and restart the kernel. Paths match your moved Dropbox folders.

Default `DROP_BOTTOM_ASSETS_PCT=0` keeps the full eligible sample and unmodified ROA. Existing extreme ROA can dominate this run. Optional predeclared values 1 and 5 create separate sensitivity runs and change the target population. Do not choose the restriction based on a favorable industry pattern. No outcome clipping, winsorization or hidden report deletion is performed.

Adjustment architecture/adversarial strength and RF nuisance settings match the previous full-sample workflow. RF is added as a **separate effect learner**, fitted with residual weights; this is not simply predicting Y. Do not call its raw regression pseudo-target an individual effect. To preserve RAM/CPU/GPU capacity, run this after or separately from other large training notebooks.


In [ ]:
from pathlib import Path
import os,json,time,hashlib,random,gc,copy
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import GroupKFold,GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from threadpoolctl import threadpool_limits
from IPython.display import display
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset,DataLoader

PROJECT_ROOT=REPO_ROOT
COMPUSTAT_ROOT=REPO_CONFIG['data_root']
EMBEDDING_ROOT=REPO_CONFIG['embedding_root']
DATA_DIR=REPO_CONFIG['analysis_data_dir']
OUTPUT_ROOT=REPO_CONFIG['output_root']/'industry_hte_llama_rf'
EMBEDDING_ENGINE='llama'
EXPECTED_CORPUS_ID=REPO_CONFIG['expected_corpus_id']
EMBEDDING_SNAPSHOT=REPO_CONFIG['embedding_snapshots'].get('llama')
NUM_COLS=['w_roa_current','w_log_assets','w_leverage','w_cash_ratio']
CAT_COLS=['fe_year','fe_industry_sic2']
# Baseline is the full eligible sample, not a silently restricted sample.
DROP_BOTTOM_ASSETS_PCT=0  # Optional predeclared sensitivity: 1 or 5; produces a separate run.
CFG={'version':'llama_adjustment_rf_hte_three_role_v1','seed':42,'dZ':64,
 'encoder_hidden':[256,128],'outcome_hidden':[128,64],'adversary_hidden':[64,32],
 'dropout':0.1,'grl_lambda':0.05,'adv_weight':0.005,'lr':0.001,'weight_decay':1e-5,
 'epochs':40,'min_epochs':10,'patience':6,'batch_size':512,
 'rf_n_estimators':300,'rf_max_depth':12,'rf_min_samples_leaf':20,'rf_max_features':0.5,
 'hte_n_estimators':300,'hte_max_depth':8,'hte_min_samples_leaf':50,
 'hte_min_weight_fraction_leaf':0.01,'hte_max_features':1.0,
 'drop_bottom_assets_pct':DROP_BOTTOM_ASSETS_PCT,
 'min_plot_companies':100,'min_plot_reports':200,'min_training_industry_companies':10,
 'min_industry_year_companies':20,'numeric_controls':NUM_COLS,'categorical_controls':CAT_COLS}
CPU_THREADS=min(4,max(1,(os.cpu_count() or 2)-1))
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(CPU_THREADS)
_blas_limit=threadpool_limits(limits=CPU_THREADS)
assert DROP_BOTTOM_ASSETS_PCT in [0,1,5]
print('Device:',DEVICE,'| CPU threads:',CPU_THREADS)
print('Fresh Adjustment + RF nuisances + RF R-learner; existing benchmark models are not required.')

def set_seed(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def atom_json(path,value):
 path=Path(path);temp=path.with_name(path.name+'.tmp')
 temp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8');os.replace(temp,path)

def atom_csv(frame,path):
 path=Path(path);temp=path.with_name(path.name+'.tmp')
 frame.to_csv(temp,index=False,compression='gzip' if path.suffix=='.gz' else None)
 os.replace(temp,path)

def r2_np(actual,predicted):
 denom=np.sum((actual-np.mean(actual))**2)
 return float(1-np.sum((actual-predicted)**2)/denom) if denom>1e-12 else np.nan


## Step 2 — Alignment, Adjustment and effect-estimation functions
The effect model minimizes `sum (Y_residual - T_residual * tau(X))**2`. For nonzero treatment residuals this is weighted regression of the pseudo-target `Y_residual / T_residual` with weights `T_residual**2`; scaling is undone before reporting. Only numerical-zero residuals are excluded from this training calculation and their count is saved. No ratio is reported as a firm-level effect.

Adjustment inputs: learned Z, four financial W, year and industry indicators. Effect modifiers X: four financial W, year and SIC2 indicators. Thus within-industry estimates may vary with company characteristics and year. X does not include T or future ROA. The conditional slope interpretation assumes a partially linear effect `tau(X)` with valid adjustment. With more general heterogeneity, the residual loss targets a treatment-residual-variance-weighted projection; do not automatically interpret industry means as unrestricted causal ATEs.

Both networks and all preprocessors fit their designated training companies only. The same current Adjustment loss and outcome-based early stopping are used; no adversarial hyperparameter tuning is performed here.


In [ ]:
def embedding_sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()

def resolve_embedding():
 if EMBEDDING_SNAPSHOT is not None:
  choices=[Path(EMBEDDING_SNAPSHOT)]
 else:
  choices=[]
  for path in (EMBEDDING_ROOT/EMBEDDING_ENGINE/EXPECTED_CORPUS_ID).rglob('latest_snapshot.json'):
   candidate=json.loads(path.read_text(encoding='utf-8'))
   if candidate.get('complete') is True and candidate.get('corpus_id')==EXPECTED_CORPUS_ID:
    choices.append(path)
 if len(choices)!=1:
  raise RuntimeError(f'Expected one completed {EMBEDDING_ENGINE} snapshot; found {len(choices)}: {choices}. Finish export or set EMBEDDING_SNAPSHOT explicitly.')
 path=choices[0];snapshot=json.loads(path.read_text(encoding='utf-8'))
 assert snapshot.get('complete') is True, 'Embedding export is incomplete.'
 assert snapshot['corpus_id']==EXPECTED_CORPUS_ID, 'Different text corpus.'
 matrix=path.parent/snapshot['report_matrix'];index=path.parent/snapshot['report_index']
 assert embedding_sha(matrix)==snapshot['report_matrix_sha256'], 'Embedding matrix checksum mismatch.'
 assert embedding_sha(index)==snapshot['report_index_sha256'], 'Embedding index checksum mismatch.'
 return path,snapshot,matrix,index

_embedding_cache={}
def aligned_report_vectors(source):
 if 'matrix' not in _embedding_cache:
  idx=pd.read_csv(SELECTED_INDEX,dtype={'doc_id':'string','corpus_id':'string'})
  matrix=np.load(SELECTED_MATRIX,mmap_mode='r',allow_pickle=False)
  assert idx.doc_id.notna().all() and idx.doc_id.is_unique, 'Missing/duplicate embedding doc_id.'
  rows=pd.to_numeric(idx.embedding_row,errors='raise').to_numpy(dtype=float)
  assert np.isfinite(rows).all() and np.equal(rows,np.floor(rows)).all()
  assert np.array_equal(np.sort(rows),np.arange(len(idx))), 'Invalid embedding row index.'
  assert matrix.shape==(len(idx),int(SELECTED_SNAPSHOT['dimension']))
  assert len(idx)==int(SELECTED_SNAPSHOT['reports_exported'])
  assert idx.corpus_id.eq(EXPECTED_CORPUS_ID).all()
  _embedding_cache.update(index=idx.set_index('doc_id'),matrix=matrix)
 idx=_embedding_cache['index'];matrix=_embedding_cache['matrix']
 missing=source.loc[~source.doc_id.isin(idx.index),'doc_id']
 assert missing.empty, f'Missing {len(missing)} report embeddings. No silent sample deletion is allowed.'
 matched=idx.loc[source.doc_id]
 assert np.allclose(pd.to_numeric(matched.competition_intensity),pd.to_numeric(source.competition_intensity),rtol=1e-8,atol=1e-10), 'Competition text corpus mismatch.'
 aligned=np.asarray(matrix[matched.embedding_row.to_numpy(dtype=np.int64)],dtype=np.float32)
 assert np.isfinite(aligned).all() and (np.linalg.norm(aligned,axis=1)>0).all()
 return aligned

def assign_roles(groups,seed):
 unique=np.random.default_rng(seed).permutation(np.unique(groups));mapping={g:i for i,g in enumerate(unique)}
 labels=np.array([mapping[g] for g in groups]);fold_ids=np.full(len(groups),-1,int)
 for k,(_,test) in enumerate(GroupKFold(n_splits=3).split(labels,groups=labels)):
  fold_ids[test]=k
 rotations=[]
 for k in range(3):
  a=np.flatnonzero(fold_ids==(k+1)%3)
  b=np.flatnonzero(fold_ids==(k+2)%3)
  c=np.flatnonzero(fold_ids==k)
  ga,gb,gc_=map(lambda ix:set(groups[ix]),[a,b,c])
  assert ga.isdisjoint(gb) and ga.isdisjoint(gc_) and gb.isdisjoint(gc_)
  rotations.append((a,b,c))
 assert np.array_equal(np.sort(np.concatenate([c for a,b,c in rotations])),np.arange(len(groups)))
 return fold_ids,rotations

def fit_x(wn,wc,indices):
 return StandardScaler().fit(wn[indices]),OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32).fit(wc[indices])

def apply_x(prep,wn,wc,indices):
 return np.column_stack([prep[0].transform(wn[indices]),prep[1].transform(wc[indices])]).astype(np.float32)

def train_rf_nuisances(x,t,y,seed):
 kwargs=dict(n_estimators=CFG['rf_n_estimators'],max_depth=CFG['rf_max_depth'],
             min_samples_leaf=CFG['rf_min_samples_leaf'],max_features=CFG['rf_max_features'],n_jobs=CPU_THREADS)
 treatment=RandomForestRegressor(random_state=seed,**kwargs).fit(x,t)
 outcome=RandomForestRegressor(random_state=seed+1000,**kwargs).fit(x,y)
 return treatment,outcome

def fit_effect_forest(x,u,v,seed):
 # Weighted least squares: sum u_i^2 * (v_i/u_i - tau(x_i))^2
 # is exactly sum (v_i - u_i*tau(x_i))^2 for retained nonzero u_i.
 # The ratio is a training pseudo-outcome, NEVER an individual-effect estimate.
 ts=max(float(np.std(u)),1e-12);ys=max(float(np.std(v)),1e-12)
 us=u/ts;vs=v/ys
 keep=np.abs(us)>1e-8  # Numerical-zero guard, not discretionary overlap trimming.
 assert keep.sum()>=max(200,4*CFG['hte_min_samples_leaf'])
 target=vs[keep]/us[keep];weight=us[keep]**2;weight/=weight.mean()
 assert np.isfinite(target).all() and np.isfinite(weight).all()
 forest=RandomForestRegressor(n_estimators=CFG['hte_n_estimators'],
  max_depth=CFG['hte_max_depth'],min_samples_leaf=CFG['hte_min_samples_leaf'],
  min_weight_fraction_leaf=CFG['hte_min_weight_fraction_leaf'],
  max_features=CFG['hte_max_features'],n_jobs=CPU_THREADS,random_state=seed)
 forest.fit(x[keep],target,sample_weight=weight)
 info={'numerical_zero_residual_rows_omitted':int((~keep).sum()),
       'treatment_residual_sd':ts,'outcome_residual_sd':ys,
       'residual_weight_effective_sample_size':float(weight.sum()**2/np.sum(weight**2))}
 return forest,ys/ts,info

def residual_slope(u,v):
 denominator=float(np.sum(u*u))
 return float(np.sum(u*v)/denominator) if denominator>1e-10 else np.nan

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, gradient):
        return -ctx.lambd * gradient, None

def mlp(in_dim, hidden, out_dim, dropout):
    layers = []
    for width in hidden:
        layers += [nn.Linear(in_dim, width), nn.ReLU(), nn.Dropout(dropout)]
        in_dim = width
    return nn.Sequential(*layers, nn.Linear(in_dim, out_dim))

class AdjustmentContinuous(nn.Module):
    def __init__(self, dH, dW, cfg):
        super().__init__()
        dZ, drop = cfg["dZ"], cfg["dropout"]
        self.encoder = mlp(dH, cfg["encoder_hidden"], dZ, drop)
        self.y_head = mlp(dZ + dW + 1, cfg["outcome_hidden"], 1, drop)
        self.t_adv = mlp(dZ, cfg["adversary_hidden"], 1, drop)

def tensor(x):
    return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def encode_batches(model, h, batch_size=2048):
    model.eval()
    return np.concatenate([
        model.encoder(tensor(h[i:i + batch_size])).cpu().numpy()
        for i in range(0, len(h), batch_size)
    ])

def fit_representation(htr, wtr, ttr, ytr, hva, wva, tva, yva, seed, cfg):
    set_seed(seed)
    model = AdjustmentContinuous(htr.shape[1], wtr.shape[1], cfg).to(DEVICE)
    main_params = list(model.encoder.parameters()) + list(model.y_head.parameters())
    adv_params = list(model.t_adv.parameters())
    main_opt = torch.optim.AdamW(main_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    adv_opt = torch.optim.AdamW(adv_params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    train_data = TensorDataset(*[torch.tensor(a, dtype=torch.float32)
                                for a in [htr, wtr, ttr, ytr]])
    loader = DataLoader(train_data, batch_size=cfg["batch_size"],
                        shuffle=True, num_workers=0)
    hv, wv, tv, yv = [tensor(a) for a in [hva, wva, tva, yva]]
    best, best_state, best_epoch, bad = float("inf"), None, None, 0
    history = []
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        train_y_sum, train_adv_sum, n_train = 0.0, 0.0, 0
        for hb, wb, tb, yb in loader:
            hb, wb, tb, yb = [a.to(DEVICE) for a in [hb, wb, tb, yb]]
            # Main update: reverse the gradient going to Z, freeze adversary weights.
            for p in adv_params:
                p.requires_grad_(False)
            main_opt.zero_grad(set_to_none=True)
            z = model.encoder(hb)
            yhat = model.y_head(torch.cat([z, wb, tb[:, None]], dim=1)).squeeze(-1)
            that = model.t_adv(GradReverse.apply(z, cfg["grl_lambda"])).squeeze(-1)
            ly, la = F.mse_loss(yhat, yb), F.mse_loss(that, tb)
            (ly + cfg["adv_weight"] * la).backward()
            nn.utils.clip_grad_norm_(main_params, 5.0)
            main_opt.step()
            # Adversary update: no gradient to encoder.
            for p in adv_params:
                p.requires_grad_(True)
            adv_opt.zero_grad(set_to_none=True)
            with torch.no_grad():
                detached_z = model.encoder(hb)
            adv_loss = F.mse_loss(model.t_adv(detached_z).squeeze(-1), tb)
            adv_loss.backward()
            nn.utils.clip_grad_norm_(adv_params, 5.0)
            adv_opt.step()
            train_y_sum += ly.item() * len(hb)
            train_adv_sum += la.item() * len(hb)
            n_train += len(hb)
        model.eval()
        with torch.no_grad():
            zv = model.encoder(hv)
            yp = model.y_head(torch.cat([zv, wv, tv[:, None]], dim=1)).squeeze(-1)
            tp = model.t_adv(zv).squeeze(-1)
            val_y = F.mse_loss(yp, yv).item()
            val_adv = F.mse_loss(tp, tv).item()
        assert np.isfinite(val_y) and np.isfinite(val_adv), "Nonfinite training loss."
        history.append({"epoch": epoch, "train_y_mse": train_y_sum / n_train,
            "train_adv_mse": train_adv_sum / n_train,
            "val_y_mse": val_y, "val_adv_mse": val_adv})
        # Only checkpoints after the minimum warm-up are eligible.
        if epoch >= cfg["min_epochs"]:
            if val_y < best - 1e-6:
                best, best_epoch, bad = val_y, epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= cfg["patience"]:
                    break
    assert best_state is not None, "epochs must be >= min_epochs"
    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(history), best_epoch


## Step 3 — Load the real analysis table and align Llama vectors
Verifies file hashes and matches report vectors by doc_id. Uses the existing SIC2 column, preserving two-digit labels. Missing vectors, nonfinite required variables or invalid identifiers stop execution rather than silently changing the sample. One row remains one firm/report period; source files remain unchanged.

Three rotating company-disjoint roles: A trains Adjustment and RF nuisance models; B supplies independent residuals for effect-model training; C evaluates effect predictions. In the next rotation the roles change. Every report is evaluated once, and its company appears in neither A nor B for its prediction. Repeated annual reports from the same company always stay together within a rotation. This is a group-held-out analysis, not a chronological forecasting split and not firm fixed effects.


In [ ]:
CODE_DIGEST='e3e5b9b7140f43112039cee27d103bd697948b207611a77c4d7ce3996ea88b73'
SELECTED_SNAPSHOT_PATH,SELECTED_SNAPSHOT,SELECTED_MATRIX,SELECTED_INDEX=resolve_embedding()
receipt=json.loads((DATA_DIR/'dataset_snapshot.json').read_text(encoding='utf-8'))
assert receipt['complete']
table_path=DATA_DIR/receipt['table'];assert embedding_sha(table_path)==receipt['table_sha256']
source=pd.read_csv(table_path,dtype={'doc_id':'string','gvkey':'string',**{c:'string' for c in CAT_COLS}},low_memory=False)
assert source.doc_id.is_unique and source.doc_id.notna().all() and source.gvkey.notna().all()
assert source[NUM_COLS+CAT_COLS].notna().all().all()
source_H=aligned_report_vectors(source)
cutoff=source.w_log_assets.quantile(DROP_BOTTOM_ASSETS_PCT/100) if DROP_BOTTOM_ASSETS_PCT else -np.inf
keep=source.w_log_assets.ge(cutoff).to_numpy()
df=source.loc[keep].reset_index(drop=True).copy();H=source_H[keep].astype(np.float32)
raw_sic=pd.to_numeric(df.fe_industry_sic2,errors='coerce')
assert raw_sic.notna().all() and raw_sic.between(1,99).all() and np.equal(raw_sic,np.floor(raw_sic)).all()
df['fe_industry_sic2']=raw_sic.astype(int).astype(str).str.zfill(2)
T=100*pd.to_numeric(df.competition_intensity).to_numpy(dtype=float)
Y=pd.to_numeric(df.roa_next_fyear).to_numpy(dtype=float)
Wnum=df[NUM_COLS].to_numpy(dtype=float);Wcat=df[CAT_COLS].astype(str).to_numpy()
groups=df.gvkey.astype(str).to_numpy()
assert all(np.isfinite(x).all() for x in [H,T,Y,Wnum])
assert np.var(T)>0 and np.var(Y)>0
fold_ids,ROTATIONS=assign_roles(groups,CFG['seed'])
versions={k:metadata.version(k) for k in ['numpy','pandas','scikit-learn','torch','joblib']}
run_identity={'config':CFG,'code':CODE_DIGEST,'device':DEVICE,'versions':versions,
 'table_sha256':receipt['table_sha256'],'report_matrix_sha256':SELECTED_SNAPSHOT['report_matrix_sha256'],
 'report_index_sha256':SELECTED_SNAPSHOT['report_index_sha256'],'embedding_settings':SELECTED_SNAPSHOT['settings']}
RUN_ID=hashlib.sha256(json.dumps(run_identity,sort_keys=True).encode()).hexdigest()[:20]
RUN_DIR=OUTPUT_ROOT/RUN_ID;RUN_DIR.mkdir(parents=True,exist_ok=True)
atom_json(RUN_DIR/'run_config.json',run_identity)
atom_csv(df[['doc_id','gvkey','fe_industry_sic2','fe_year']].assign(evaluation_fold=fold_ids+1),RUN_DIR/'company_role_folds.csv')
print('Reports:',len(df),'| companies:',len(np.unique(groups)),'| Llama matrix:',H.shape)
print('Asset exclusion:',DROP_BOTTOM_ASSETS_PCT,'% | Original input data unchanged.')
print('Absolute future ROA above 100%:',int(np.sum(np.abs(Y)>1)))
print('Output:',RUN_DIR)


## Step 4 — Train / resume fresh Llama Adjustment-RF and heterogeneous-effect RF
Long-running step: three Adjustment models, six RF nuisance models and three weighted RF effect models. All years and supported industries share the effect learner; the code does not train a separate tiny RF within each industry.

No old OOF residuals are recycled. A is independent of B/C, and the effect learner never trains on its C evaluation companies. Group-held-out evaluation remains valid as a data-separation check, but is not a proof of causal identification. Predictive heterogeneity is assessed using C's residual loss against a constant slope learned on B. Positive held-out R-loss gain indicates improved prediction of residual outcomes, not a formal causal heterogeneity test.

Complete rotations save models, scalers, predictions and a receipt. Restart and rerun Steps 1–4 with unchanged inputs/settings to reuse completed rotations. An interrupted rotation retrains from its start; no per-epoch recovery is promised. Allow only one writer for a run folder.


In [ ]:
def run_rotation(rotation,a,b,c):
 folder=RUN_DIR/f'rotation_{rotation}';folder.mkdir(parents=True,exist_ok=True)
 result_path=folder/'evaluation_predictions.csv.gz';ready=folder/'complete.json'
 if ready.is_file():
  saved=pd.read_csv(result_path,dtype={'doc_id':'string','gvkey':'string','fe_industry_sic2':'string'})
  assert np.array_equal(saved.doc_id.astype(str).to_numpy(),df.iloc[c].doc_id.astype(str).to_numpy())
  print('Reused completed rotation:',rotation)
  return saved,json.loads((folder/'diagnostics.json').read_text())
 print(f'Rotation {rotation}/3: A={len(a)} nuisance training; B={len(b)} effect training; C={len(c)} evaluation.',flush=True)
 # A only: fit neural preprocessing, train Adjustment, select epoch with a firm-held-out validation subset.
 split=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=CFG['seed']+rotation)
 it,iv=next(split.split(H[a],groups=groups[a]));itr,iva=a[it],a[iv]
 assert set(groups[itr]).isdisjoint(groups[iva])
 hs=StandardScaler().fit(H[itr]);wp=fit_x(Wnum,Wcat,itr)
 ts=StandardScaler().fit(T[itr,None]);ys=StandardScaler().fit(Y[itr,None])
 model,history,epoch=fit_representation(
  hs.transform(H[itr]),apply_x(wp,Wnum,Wcat,itr),ts.transform(T[itr,None]).ravel(),ys.transform(Y[itr,None]).ravel(),
  hs.transform(H[iva]),apply_x(wp,Wnum,Wcat,iva),ts.transform(T[iva,None]).ravel(),ys.transform(Y[iva,None]).ravel(),
  CFG['seed']+rotation,CFG)
 zscale=StandardScaler().fit(encode_batches(model,hs.transform(H[itr])))
 def adjustment(indices):
  z=zscale.transform(encode_batches(model,hs.transform(H[indices])))
  return np.column_stack([z,apply_x(wp,Wnum,Wcat,indices)]).astype(np.float32)
 xa,xb,xc=[adjustment(ix) for ix in [a,b,c]]
 print('  Adjustment selected epoch:',epoch,'| Fit RF treatment/outcome nuisances on A.',flush=True)
 treatment,outcome=train_rf_nuisances(xa,T[a],Y[a],CFG['seed']+rotation)
 tb,yb=treatment.predict(xb),outcome.predict(xb)
 tc,yc=treatment.predict(xc),outcome.predict(xc)
 ub,vb=T[b]-tb,Y[b]-yb;uc,vc=T[c]-tc,Y[c]-yc
 # B only: effect modifiers are real financial controls, year, and SIC2; no encoder coordinates are pooled.
 xp=fit_x(Wnum,Wcat,b);effect_xb=apply_x(xp,Wnum,Wcat,b);effect_xc=apply_x(xp,Wnum,Wcat,c)
 print('  Fit weighted RF R-learner on independent B residuals.',flush=True)
 forest,scale,weight_info=fit_effect_forest(effect_xb,ub,vb,CFG['seed']+rotation+500)
 cate=forest.predict(effect_xc)*scale
 assert np.isfinite(cate).all()
 constant_b=residual_slope(ub,vb)
 residual_loss=np.mean((vc-uc*cate)**2);constant_loss=np.mean((vc-uc*constant_b)**2)
 support=df.iloc[b].groupby('fe_industry_sic2').gvkey.nunique()
 required=CFG['min_training_industry_companies']
 supported=df.iloc[c].fe_industry_sic2.map(support).fillna(0).ge(required).to_numpy()
 out=df.iloc[c][['doc_id','gvkey','fe_year','fe_industry_sic2',*NUM_COLS]].copy()
 out['evaluation_fold']=rotation;out['T_pp']=T[c];out['Y_roa']=Y[c]
 out['T_hat']=tc;out['Y_hat']=yc;out['T_residual']=uc;out['Y_residual']=vc
 out['estimated_slope_roa_decimal_per_T_pp']=cate
 out['estimated_effect_roa_pp_per_T_pp']=100*cate
 out['industry_supported_in_effect_training']=supported
 out['constant_slope_trained_on_B_roa_pp']=100*constant_b
 diagnostics={'rotation':rotation,'N_A':len(a),'N_B':len(b),'N_C':len(c),
  'companies_A':len(np.unique(groups[a])),'companies_B':len(np.unique(groups[b])),
  'companies_C':len(np.unique(groups[c])),'selected_epoch':int(epoch),
  'R2_T_evaluation':r2_np(T[c],tc),'R2_Y_evaluation':r2_np(Y[c],yc),
  'R_loss_heterogeneous':float(residual_loss),'R_loss_constant':float(constant_loss),
  'R_loss_gain_fraction':float(1-residual_loss/constant_loss) if constant_loss>0 else 0.,
  'evaluation_t_residual_variance_ratio':float(np.mean(uc**2)/np.var(T[c])),
  'evaluation_industry_supported_share':float(supported.mean()),**weight_info}
 # Save model artifacts before publishing the role completion receipt.
 history.to_csv(folder/'Adjustment_training_history.csv',index=False)
 torch.save({'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},'config':CFG},folder/'adjustment_encoder.pt')
 joblib.dump({'H':hs,'W':wp,'T':ts,'Y':ys,'Z':zscale,'effect_X':xp,'effect_scale':scale},folder/'preprocessors.joblib')
 joblib.dump({'treatment':treatment,'outcome':outcome},folder/'rf_nuisances.joblib')
 joblib.dump(forest,folder/'rf_effect_model.joblib')
 feature_names=NUM_COLS+list(xp[1].get_feature_names_out(CAT_COLS))
 atom_csv(pd.DataFrame({'feature':feature_names,'importance':forest.feature_importances_}),folder/'effect_modifier_importance.csv')
 atom_csv(out,result_path);atom_json(folder/'diagnostics.json',diagnostics)
 atom_json(ready,{'complete':True,'rotation':rotation,'reports':len(c)})
 del model,forest,treatment,outcome;gc.collect()
 if torch.cuda.is_available():torch.cuda.empty_cache()
 print('Completed rotation:',rotation,flush=True)
 return out,diagnostics

started=time.perf_counter();pieces=[];diagnostic_rows=[]
for rotation,(a,b,c) in enumerate(ROTATIONS,1):
 result,diagnostic=run_rotation(rotation,a,b,c)
 pieces.append(result);diagnostic_rows.append(diagnostic)
hte=pd.concat(pieces,ignore_index=True)
assert len(hte)==len(df) and hte.doc_id.is_unique
assert set(hte.doc_id.astype(str))==set(df.doc_id.astype(str))
# Reorder by the original analysis table, not by model-folder or CSV order.
hte=hte.set_index('doc_id').loc[df.doc_id].reset_index()
atom_csv(hte,RUN_DIR/'report_hte_predictions.csv.gz')
atom_csv(pd.DataFrame(diagnostic_rows),RUN_DIR/'heldout_diagnostics.csv')
atom_json(RUN_DIR/'predictions_complete.json',{'complete':True,'reports':len(hte),'rotations':3})
print('Elapsed minutes:',round((time.perf_counter()-started)/60,1))
display(pd.DataFrame(diagnostic_rows))


## Step 5 — Industry effect summaries and diagnostics
Main effect summaries/distributions weight annual report rows equally, consistent with the empirical analysis unit. Company-equal-weight means are separately exported. Report averages of forest predictions and industry residual slopes are different quantities; both are exported and should not be labeled interchangeably as ATE.

No model-based 95% CATE intervals or industry difference p-values are claimed: repeated-firm dependence, learned representations and full-pipeline uncertainty require additional inference work. The distribution percentiles below describe the fitted conditional effects, not certainty about real causal effects. Small/poorly represented industries remain in the CSV with flags, but do not enter main plots by default.


In [ ]:
EFFECT_COL='estimated_effect_roa_pp_per_T_pp'

def industry_summary(frame):
 rows=[]
 for code,part in frame.groupby('fe_industry_sic2',sort=True):
  effects=part[EFFECT_COL].to_numpy(dtype=float)
  support=part.industry_supported_in_effect_training.astype(bool).mean()
  years=pd.to_numeric(part.fe_year,errors='raise')
  rows.append({'sic2':str(code).zfill(2),'n_reports':len(part),'n_companies':part.gvkey.nunique(),
   'mean_estimated_effect':float(effects.mean()),'median_estimated_effect':float(np.median(effects)),
   'q10':float(np.quantile(effects,.10)),'q25':float(np.quantile(effects,.25)),
   'q75':float(np.quantile(effects,.75)),'q90':float(np.quantile(effects,.90)),
   'sd_estimated_effect':float(np.std(effects,ddof=1)) if len(effects)>1 else 0.,
   'positive_estimated_share':float(np.mean(effects>0)),
   'firm_equal_weight_mean':float(part.groupby('gvkey')[EFFECT_COL].mean().mean()),
   'industry_residual_slope_roa_pp':100*residual_slope(part.T_residual.to_numpy(),part.Y_residual.to_numpy()),
   'mean_T_pp':float(part.T_pp.mean()),'absolute_Y_above_100pct_share':float(np.mean(np.abs(part.Y_roa)>1)),
   'min_year':int(years.min()),'max_year':int(years.max()),
   'share_supported_in_effect_training':float(support),
   'eligible_for_main_plots':bool(part.gvkey.nunique()>=CFG['min_plot_companies']
     and len(part)>=CFG['min_plot_reports'] and support>=1-1e-10)})
 return pd.DataFrame(rows)

summary=industry_summary(hte)
atom_csv(summary,RUN_DIR/'industry_hte_summary.csv')
company_effects=hte.groupby(['fe_industry_sic2','gvkey'],as_index=False).agg(
 mean_estimated_effect=(EFFECT_COL,'mean'),n_reports=('doc_id','size'))
atom_csv(company_effects,RUN_DIR/'company_within_industry_effects.csv')
# The main summaries/plots are report-weighted; firm-equal-weight means are exported for comparison.
yearly=hte.groupby(['fe_industry_sic2','fe_year'],as_index=False).agg(
 mean_estimated_effect=(EFFECT_COL,'mean'),median_estimated_effect=(EFFECT_COL,'median'),
 n_reports=('doc_id','size'),n_companies=('gvkey','nunique'))
yearly['year']=pd.to_numeric(yearly.fe_year,errors='raise').astype(int)
yearly['supported_year_cell']=yearly.n_companies.ge(CFG['min_industry_year_companies'])
atom_csv(yearly,RUN_DIR/'industry_year_hte_trends.csv')

pooled=100*residual_slope(hte.T_residual.to_numpy(),hte.Y_residual.to_numpy())
overall={'reports':len(hte),'companies':hte.gvkey.nunique(),
 'mean_estimated_effect_roa_pp_per_T_pp':float(hte[EFFECT_COL].mean()),
 'median_estimated_effect_roa_pp_per_T_pp':float(hte[EFFECT_COL].median()),
 'pooled_residual_slope_roa_pp_per_T_pp':float(pooled),
 'industries':len(summary),'industries_eligible_for_main_plots':int(summary.eligible_for_main_plots.sum()),
 'status':'exploratory_conditional_slope_estimates_not_validated_causal_effects',
 'interval_note':'Distribution quantiles are not confidence intervals; no full-pipeline inferential intervals are claimed.'}
atom_json(RUN_DIR/'overall_hte_summary.json',overall)
print(json.dumps(overall,indent=2))
display(summary[['sic2','n_companies','n_reports','mean_estimated_effect',
                 'median_estimated_effect','q25','q75','eligible_for_main_plots']].round(4))
# Held-out effect-bin calibration is a diagnostic, not proof of heterogeneous causal effects.
if hte[EFFECT_COL].nunique()>1:
 temp=hte.copy();temp['effect_bin']=pd.qcut(temp[EFFECT_COL],q=5,duplicates='drop')
 bins=[]
 for label,part in temp.groupby('effect_bin',observed=True):
  bins.append({'bin':str(label),'n_reports':len(part),'n_companies':part.gvkey.nunique(),
   'mean_predicted_effect':part[EFFECT_COL].mean(),
   'residual_slope_in_bin':100*residual_slope(part.T_residual.to_numpy(),part.Y_residual.to_numpy())})
 atom_csv(pd.DataFrame(bins),RUN_DIR/'heldout_effect_bin_diagnostic.csv')


## Step 6 — Industry comparison, within-industry distributions and annual patterns
Outputs PNG and vector PDF. Main plot thresholds: at least 100 companies and 200 reports, with at least 10 companies from the industry in each applicable effect-training role. These are practical support screens, not guarantees of adequate overlap.

1. Industry mean dot with 10–90% and 25–75% estimated-effect ranges, explicitly **not confidence intervals**.
2. One violin per SIC2 industry (median/IQR inside).
3. Separate histogram for every supported industry, with mean/median and sample counts.
4. Annual patterns for the eight largest supported industries; all industry/year means remain in CSV. Years with fewer than 20 companies are left as gaps. These plots display temporal patterns, not causal time-trend estimates.

All plotted effect values are retained; there is no visual tail trimming. Industries are ordered by code, or selected for the time panels by company count, never by sign or significance.


In [ ]:
from io import BytesIO
FIG_DIR=RUN_DIR/'figures';FIG_DIR.mkdir(exist_ok=True)

def save_figure(fig,name):
 for extension in ['png','pdf']:
  buffer=BytesIO();fig.savefig(buffer,format=extension,dpi=200,bbox_inches='tight',facecolor='white')
  (FIG_DIR/f'{name}.{extension}').write_bytes(buffer.getvalue())
 plt.show();plt.close(fig)

shown=summary.loc[summary.eligible_for_main_plots].sort_values('sic2').copy()
print('Industry plot threshold:',CFG['min_plot_companies'],'companies and',CFG['min_plot_reports'],'reports; adequate industry training support required.')
print('All industries remain in the CSV; plot exclusions are flagged explicitly.')
if shown.empty:
 print('No industries meet the plot thresholds. Review industry_hte_summary.csv; no observations were silently deleted.')
else:
 # Figure 1: mean estimated effects with distribution ranges (NOT uncertainty intervals).
 for page,start in enumerate(range(0,len(shown),20),1):
  part=shown.iloc[start:start+20].reset_index(drop=True);yy=np.arange(len(part))
  fig,ax=plt.subplots(figsize=(10,max(5,0.38*len(part)+2)))
  ax.hlines(yy,part.q10,part.q90,color='#B8C9D7',linewidth=2,label='10–90% estimated-effect range')
  ax.hlines(yy,part.q25,part.q75,color='#147D92',linewidth=5,label='25–75% estimated-effect range')
  ax.scatter(part.mean_estimated_effect,yy,color='#D55E00',s=28,zorder=3,label='Report-weighted mean')
  ax.axvline(0,color='black',linestyle='--',linewidth=1)
  ax.set_yticks(yy);ax.set_yticklabels([f'SIC {r.sic2} | firms={r.n_companies:,}; reports={r.n_reports:,}' for r in part.itertuples()])
  ax.invert_yaxis();ax.set_xlabel('Estimated ROA percentage-point change per +1 pp competition language')
  ax.set_title('Industry effect summaries — Llama Adjustment-RF + RF R-learner\nExploratory; bars describe estimated-effect distributions, not confidence intervals')
  ax.legend(loc='best',fontsize=8);ax.grid(axis='x',alpha=.2);fig.tight_layout()
  save_figure(fig,f'industry_effects_page_{page:02d}')

 # Figure 2: one violin per industry; no individual residual ratio is plotted.
 for page,start in enumerate(range(0,len(shown),12),1):
  part=shown.iloc[start:start+12];codes=part.sic2.tolist()
  values=[hte.loc[hte.fe_industry_sic2.eq(code),EFFECT_COL].to_numpy() for code in codes]
  fig,ax=plt.subplots(figsize=(max(10,len(codes)*.85),5.5))
  for j,val in enumerate(values,1):
   if np.ptp(val)>1e-10:
    violin=ax.violinplot([val],positions=[j],showextrema=False,widths=.8)
    for body in violin['bodies']:body.set_facecolor('#147D92');body.set_alpha(.55)
   q25,median,q75=np.quantile(val,[.25,.5,.75])
   ax.vlines(j,q25,q75,color='#16324F',linewidth=4);ax.scatter(j,median,color='white',edgecolor='#16324F',s=24,zorder=3)
  ax.axhline(0,color='black',linestyle='--',linewidth=1)
  ax.set_xticks(range(1,len(codes)+1));ax.set_xticklabels(['SIC '+c for c in codes])
  ax.set_ylabel('Estimated effect: ROA pp per +1 pp competition language')
  ax.set_title('Within-industry distribution of held-out conditional-effect estimates\nWhite dot = median; thick line = interquartile range; not a confidence interval')
  ax.grid(axis='y',alpha=.2);fig.tight_layout();save_figure(fig,f'industry_violins_page_{page:02d}')

 # Figure 3: a separate histogram underneath an effect summary for EVERY supported industry.
 for page,start in enumerate(range(0,len(shown),4),1):
  part=shown.iloc[start:start+4];fig,axes=plt.subplots(2,2,figsize=(12,8))
  for ax,row in zip(axes.flat,part.itertuples()):
   val=hte.loc[hte.fe_industry_sic2.eq(row.sic2),EFFECT_COL].to_numpy()
   ax.hist(val,bins=30,density=True,color='#147D92',alpha=.75,edgecolor='white')
   ax.axvline(0,color='black',linestyle='--',linewidth=1,label='Zero')
   ax.axvline(row.mean_estimated_effect,color='#D55E00',linewidth=1.7,label='Mean')
   ax.set_title(f'SIC {row.sic2} | mean={row.mean_estimated_effect:.3f}; median={row.median_estimated_effect:.3f}\n{row.n_companies:,} firms; {row.n_reports:,} reports')
   ax.set_xlabel('Estimated ROA pp per +1 pp competition language');ax.set_ylabel('Density');ax.legend(fontsize=8)
  for ax in axes.flat[len(part):]:ax.set_visible(False)
  fig.suptitle('Within-industry estimated-effect distributions — exploratory')
  fig.tight_layout();save_figure(fig,f'industry_distributions_page_{page:02d}')

 # Figure 4: chronological patterns are separate from cross-industry distributions.
 codes=shown.sort_values('n_companies',ascending=False).head(8).sic2.tolist()
 fig,axes=plt.subplots(2,4,figsize=(16,7),squeeze=False)
 for ax,code in zip(axes.flat,codes):
  part=yearly.loc[yearly.fe_industry_sic2.eq(code)].sort_values('year').copy()
  # Gaps remain gaps; a small year-cell is not converted into a zero effect.
  full_years=np.arange(yearly.year.min(),yearly.year.max()+1)
  series=part.set_index('year').mean_estimated_effect.where(part.set_index('year').supported_year_cell).reindex(full_years)
  ax.plot(series.index,series.values,marker='o',markersize=3,color='#147D92')
  ax.axhline(0,color='black',linestyle='--',linewidth=.8)
  ax.set_title('SIC '+code);ax.set_xlabel('Report fiscal year');ax.set_ylabel('Mean estimated effect');ax.grid(alpha=.2)
 for ax in axes.flat[len(codes):]:ax.set_visible(False)
 fig.suptitle('Annual patterns in estimated effects: eight largest supported industries\nYear-cells below the company threshold are omitted; these are not time-series causal trend estimates.')
 fig.tight_layout();save_figure(fig,'industry_year_trends_largest8')

atom_json(RUN_DIR/'complete.json',{'complete':True,'reports':len(hte),'industries':len(summary),
 'supported_industries':len(shown),'status':'exploratory','figures_folder':str(FIG_DIR)})
print('FINISHED. Send back industry_hte_summary.csv, heldout_diagnostics.csv, overall_hte_summary.json and a few figures.')
print('Output:',RUN_DIR)


## What to send back
Send `industry_hte_summary.csv`, `heldout_diagnostics.csv`, `overall_hte_summary.json`, and the industry-effect/violin plots first. Keep `report_hte_predictions.csv.gz` and the model folders locally. A distribution crossing zero means the model predicts both signs; it does not establish statistically significant within-industry causal differences.

Method reference: Nie and Wager, *Quasi-Oracle Estimation of Heterogeneous Treatment Effects*, https://arxiv.org/abs/1712.04912. This notebook uses a continuous-treatment partially linear residual-loss extension and three company-disjoint roles; it is not a claim of a validated full-panel Adjustment inference procedure.
